# FraudShield TRL + Unsloth Colab

End-to-end training notebook for the FraudShield Theme 3.1 environment.

This notebook installs the repo, builds a state bank from real environment states, trains a small LoRA policy with GRPO, evaluates the heuristic baseline versus the locally saved trained model, and writes the training artifacts expected for submission.


In [ ]:
%pip install -q openenv-core trl unsloth transformers datasets peft accelerate
%cd /content
!rm -rf Fraudshield
!git clone https://github.com/DevikaJ2005/Fraudshield.git
%cd /content/Fraudshield
!ls
%pip install -q -e .


In [ ]:
import copy
import json
import os
import subprocess
from collections import defaultdict

from datasets import Dataset

from fraudshield_env import FraudShieldEnvironment
from llm_agent import SnapshotCalibratedFraudDetectionAgent
from models import ActionTypeEnum, FraudCheckAction, ResolutionEnum

env = FraudShieldEnvironment(data_path='data', seed=42)
assert env.load_data(), 'FraudShield snapshot failed to load.'
print('FraudShield loaded:', env.data_loader.get_bundle_summary())

def serialize_observation(observation):
    return json.dumps(
        {
            'case_id': observation.case_id,
            'task_name': observation.task_name.value,
            'current_screen': observation.current_screen.value,
            'visible_panels': observation.visible_panels,
            'revealed_evidence': observation.revealed_evidence,
            'linked_case_ids': observation.linked_case_ids,
            'remaining_steps': observation.remaining_steps,
            'remaining_sla': observation.remaining_sla,
            'note_required': observation.note_required,
            'allowed_actions': [action.value for action in observation.allowed_actions],
            'case_summary': observation.case_summary.model_dump(mode='json'),
            'app_context': observation.app_context,
        },
        ensure_ascii=True,
        separators=(',', ':'),
    )

def prompt_from_observation(observation):
    return (
        'You are a fraud analyst working in a simulated investigation workflow. '
        'Return JSON only with keys action_type, investigation_target, decision, confidence, reasoning. '
        'Use action_type investigate or decide.\nObservation: '
        f"{serialize_observation(observation)}\n"
    )

class FraudShieldTextEnv:
    def __init__(self, task_name='easy'):
        self.task_name = task_name
        self.env = FraudShieldEnvironment(data_path='data', seed=42)
        self.env.load_data()
        self.invalid_outputs = 0

    def reset(self):
        reset_result = self.env.reset(self.task_name)
        return prompt_from_observation(reset_result.observation)

    def step(self, action):
        result = self.env.step(action)
        return prompt_from_observation(result.observation), result.reward.value, result.done


In [ ]:
STATE_BANK = {}
INVALID_OUTPUTS = defaultdict(int)

def parse_json_payload(text):
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1:
        raise ValueError('No JSON object found in model output.')
    return json.loads(text[start:end+1])

def map_investigation_target(alias):
    mapping = {
        'merchant_profile': ActionTypeEnum.FETCH_MERCHANT_PROFILE,
        'customer_profile': ActionTypeEnum.FETCH_CUSTOMER_PROFILE,
        'network_graph': ActionTypeEnum.FETCH_NETWORK_GRAPH,
        'device_intel': ActionTypeEnum.FETCH_NETWORK_GRAPH,
        'payment_trace': ActionTypeEnum.REVIEW_TRANSACTION,
        'fulfillment_review': ActionTypeEnum.REVIEW_TRANSACTION,
        'policy_review': ActionTypeEnum.CHECK_POLICY,
        'trust_notes': ActionTypeEnum.CHECK_POLICY,
    }
    if alias not in mapping:
        raise ValueError(f'Unsupported investigation_target: {alias}')
    return mapping[alias]

def map_decision_to_resolution(decision, confidence, observation, case_id):
    decision = str(decision).strip().lower()
    confidence = max(0.0, min(1.0, float(confidence)))
    if observation.note_required:
        return FraudCheckAction(
            case_id=case_id,
            action_type=ActionTypeEnum.ADD_CASE_NOTE,
            note_text='Decision summary: routing based on the visible investigation evidence.'
        )
    if decision == 'legitimate':
        resolution = ResolutionEnum.APPROVE if confidence >= 0.75 or observation.task_name.value == 'easy' else ResolutionEnum.REQUEST_DOCS
    elif decision == 'fraud':
        network = observation.revealed_evidence.get('network_graph', {}).get('facts', {})
        if observation.task_name.value == 'hard' and case_id.endswith('primary') and network.get('cluster_alert_score', 0.0) >= 0.75 and network.get('linked_case_ids'):
            resolution = ResolutionEnum.ESCALATE
        elif confidence < 0.70:
            resolution = ResolutionEnum.HOLD
        else:
            resolution = ResolutionEnum.BLOCK
    else:
        raise ValueError(f'Unsupported decision: {decision}')
    return FraudCheckAction(
        case_id=case_id,
        action_type=ActionTypeEnum.RESOLVE_CASE,
        resolution=resolution,
        reasoning='Routing based on the visible investigation evidence and current confidence.'
    )

def payload_to_public_action(payload, observation, case_id):
    action_type = str(payload.get('action_type', '')).strip().lower()
    if action_type == 'investigate':
        mapped_action = map_investigation_target(str(payload.get('investigation_target', '')).strip().lower())
        return FraudCheckAction(
            case_id=case_id,
            action_type=mapped_action,
            reasoning='Investigation selected from the serialized observation state.'
        )
    if action_type == 'decide':
        return map_decision_to_resolution(payload.get('decision'), payload.get('confidence', 0.5), observation, case_id)
    raise ValueError(f'Unsupported action_type: {action_type}')

def build_state_bank(per_task_limit=24):
    heuristic = SnapshotCalibratedFraudDetectionAgent()
    records = []
    for task_name in ('easy', 'medium', 'hard'):
        env = FraudShieldEnvironment(data_path='data', seed=42)
        env.load_data()
        observation = env.reset(task_name).observation
        while not env.is_done and len([r for r in records if r['task_name'] == task_name]) < per_task_limit:
            state_id = f'{task_name}_{len(records):04d}'
            STATE_BANK[state_id] = copy.deepcopy(env)
            records.append({'prompt': prompt_from_observation(observation), 'state_id': state_id, 'task_name': task_name})
            action = heuristic.decide(observation)
            observation = env.step(action).observation
    return Dataset.from_list(records)

train_dataset = build_state_bank(per_task_limit=24)
train_dataset

def env_reward_from_completion(state_id, completion_text):
    format_reward = 0.0
    try:
        payload = parse_json_payload(completion_text)
        format_reward += 0.1
        if payload.get('action_type') in {'investigate', 'decide'}:
            format_reward += 0.1
        env_copy = copy.deepcopy(STATE_BANK[state_id])
        observation = env_copy._build_observation()
        action = payload_to_public_action(payload, observation, observation.case_id)
        result = env_copy.step(action)
        return result.reward.value + format_reward
    except Exception:
        INVALID_OUTPUTS[state_id] += 1
        return -0.2


In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
)
print('Loaded base model and LoRA adapters.')


In [ ]:
from trl import GRPOConfig, GRPOTrainer

def reward_func(prompts, completions, state_id, **kwargs):
    rewards = []
    for completion, current_state_id in zip(completions, state_id):
        if isinstance(completion, list):
            completion_text = completion[0].get('content', '')
        else:
            completion_text = str(completion)
        rewards.append(env_reward_from_completion(current_state_id, completion_text))
    return rewards

training_args = GRPOConfig(
    output_dir='fraudshield-grpo-run',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    max_prompt_length=2048,
    max_completion_length=256,
    num_generations=4,
    logging_steps=1,
    save_strategy='no',
)

def task_subset(task_name):
    return train_dataset.filter(lambda row: row['task_name'] == task_name)

easy_dataset = task_subset('easy')
medium_dataset = task_subset('medium')
hard_dataset = task_subset('hard')


In [ ]:
trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_func,
    args=training_args,
    train_dataset=easy_dataset,
    processing_class=tokenizer,
)
trainer.train()

trainer.train_dataset = medium_dataset
trainer.train()

trainer.train_dataset = hard_dataset
trainer.train()

OUTPUT_DIR = 'trained_policy'
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved trained policy to', OUTPUT_DIR)


In [ ]:
import shutil

def run_inference(extra_env=None):
    env_vars = os.environ.copy()
    if extra_env:
        env_vars.update(extra_env)
    completed = subprocess.run(
        ['python', 'inference.py'],
        capture_output=True,
        text=True,
        env=env_vars,
        check=True,
    )
    with open('fraudshield_baseline_results.json', 'r', encoding='utf-8') as handle:
        return json.load(handle), completed.stdout

baseline_results, baseline_stdout = run_inference()
trained_results, trained_stdout = run_inference({'LOCAL_MODEL_PATH': OUTPUT_DIR})

comparison_rows = []
for task_name in ('easy', 'medium', 'hard'):
    comparison_rows.append({
        'task': task_name,
        'heuristic_score': baseline_results[task_name]['score'],
        'trained_score': trained_results[task_name]['score'],
        'delta': trained_results[task_name]['score'] - baseline_results[task_name]['score'],
    })

print(json.dumps(comparison_rows, indent=2))


In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
reward_points = []
loss_points = []

for entry in log_history:
    step = entry.get('step')
    if step is None:
        continue
    reward_value = None
    for key in ('reward', 'rewards/mean', 'objective/rlhf_reward', 'objective/reward'):
        if key in entry:
            reward_value = entry[key]
            break
    if reward_value is not None:
        reward_points.append((step, reward_value))
    if 'loss' in entry:
        loss_points.append((step, entry['loss']))

if reward_points:
    xs, ys = zip(*reward_points)
    plt.figure(figsize=(8, 4))
    plt.plot(xs, ys)
    plt.xlabel('training step')
    plt.ylabel('reward')
    plt.title('FraudShield reward curve')
    plt.tight_layout()
    plt.savefig('reward_curve.png')
    plt.close()

if loss_points:
    xs, ys = zip(*loss_points)
    plt.figure(figsize=(8, 4))
    plt.plot(xs, ys)
    plt.xlabel('training step')
    plt.ylabel('loss')
    plt.title('FraudShield loss curve')
    plt.tight_layout()
    plt.savefig('loss_curve.png')
    plt.close()

summary = {
    'status': 'completed',
    'model_name': MODEL_NAME,
    'local_model_path': OUTPUT_DIR,
    'baseline_final_score': baseline_results['final_score'],
    'trained_final_score': trained_results['final_score'],
    'score_delta': trained_results['final_score'] - baseline_results['final_score'],
    'task_comparison': comparison_rows,
    'invalid_output_counts': dict(INVALID_OUTPUTS),
    'artifacts': {
        'reward_curve': 'reward_curve.png',
        'loss_curve': 'loss_curve.png',
        'trained_model_dir': OUTPUT_DIR,
    },
}

with open('training_summary.json', 'w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)

print(json.dumps(summary, indent=2))
